# ✈️ Phase 4 — Machine Learning Model, Results & Discussion

**Course:** COMP4381 Data Science and Analytics  
**Instructor:** Ahmed Sabbah  
**Team:** Amro Muzahem  
**Dataset:** Cleaned flight punctuality data — 6 European countries (FR, DE, IT, ES, NL, GB)  
**Target Variable:** `Is_Delayed` — Binary Classification (1 = Delayed, 0 = On Time)

---

This is a **binary classification** problem: we want to predict whether an airport-day record will be classified as delayed (1) or on time (0), where delay is defined as an average schedule delay exceeding 15 minutes.

We will:
1. Load the cleaned dataset from Phase 3
2. Preprocess it using a `ColumnTransformer` pipeline (as taught in class)
3. Fit a **Logistic Regression** model
4. Evaluate using accuracy, precision, recall, F1-score, confusion matrix, and classification report
5. Check for underfit or overfit by comparing train vs. test metrics
6. Discuss results and limitations


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn import set_config
set_config(transform_output='pandas')
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
)

print("Libraries loaded successfully.")


## 2. Load the Cleaned Dataset

We load `cleaned_flight_data.csv` produced in Phase 3.  
This dataset has already been merged, cleaned, and enriched with engineered features.


In [ ]:
df = pd.read_csv('../data/processed/cleaned_flight_data.csv')

print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


## 2b. Clean the Punctuality Percentage Columns

`Departure Punctuality %`, `Arrival Punctuality %`, and `Operated Schedules %` arrive from the source as **text strings with a trailing `%`** (e.g. `"92.4%"`), so pandas loads them as `object` dtype rather than numeric.

Rather than dropping them, we strip the `%` sign and convert them to `float`. Once they are numeric, the existing `ColumnTransformer`'s `num_selector = make_column_selector(dtype_include='number')` will pick them up **automatically** — no other pipeline changes are needed.

These are genuinely useful punctuality signals, so keeping them gives the model more meaningful features instead of throwing away information just because of formatting.


In [ ]:
pct_cols = ['Departure Punctuality %', 'Arrival Punctuality %', 'Operated Schedules %']

for col in pct_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.strip()
            .replace({'nan': np.nan, '': np.nan})
            .astype(float)
        )

print("Dtypes after cleaning:")
print(df[pct_cols].dtypes)
print("\nSample values:")
df[pct_cols].head()


## 3. Define Features (X) and Target (y)

The **target** is `Is_Delayed` — a binary column created in Phase 3 (1 = delayed, 0 = on time).

We drop columns that would cause problems or leak information:

| Column dropped | Reason |
|---|---|
| `Date` | Identifier — the model should not memorize specific dates |
| `Airport`, `municipality` | Text identifiers already summarized by `iso_country` |
| `DayOfWeek` | Already captured by the binary `Weekend` column |
| `AverageDelay` | This is what `Is_Delayed` was derived from — including it is **data leakage** |
| `Is_Delayed` | This is the target (y), not a feature |

`Departure Punctuality %`, `Arrival Punctuality %`, and `Operated Schedules %` are **kept** — they were cleaned to numeric in step 2b above and are useful, non-leaking predictors.


In [ ]:
drop_cols = [
    'Date', 'Airport', 'municipality', 'DayOfWeek',
    'AverageDelay',
    'Is_Delayed',
]

X = df.drop(columns=drop_cols)
y = df['Is_Delayed']

print(f"Feature matrix shape : {X.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass percentages:")
print((y.value_counts(normalize=True) * 100).round(1))


## 4. Train / Test Split

We split the data into **80% training** and **20% testing**.

- The model learns from the training set only.
- The test set is held back and used only to evaluate final performance.
- `random_state=42` makes the result reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
)

print(f"Training set : {X_train.shape[0]:,} rows")
print(f"Testing set  : {X_test.shape[0]:,} rows")


## 5. Preprocessing Pipeline

We use a `ColumnTransformer` to handle numeric and categorical columns separately — exactly as taught in class.

**Numeric pipeline:**
- `SimpleImputer(strategy='mean')` — fills any missing values with the mean
- `StandardScaler()` — scales features to have mean = 0 and std = 1

**Categorical pipeline:**
- `SimpleImputer(strategy='most_frequent')` — fills missing values with the most common category
- `OneHotEncoder(sparse_output=False, handle_unknown='ignore')` — converts categories to binary columns


In [ ]:
# ── Numeric pipeline ──────────────────────────────────────────────
num_selector = make_column_selector(dtype_include='number')

mean_imputer = SimpleImputer(strategy='mean')
scaler         = StandardScaler()
num_pipe       = make_pipeline(mean_imputer, scaler)
num_tuple      = ('numeric', num_pipe, num_selector)

# ── Categorical pipeline ───────────────────────────────────────────
cat_selector = make_column_selector(dtype_include='object')

freq_imputer = SimpleImputer(strategy='most_frequent')
ohe          = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_pipe     = make_pipeline(freq_imputer, ohe)
cat_tuple    = ('categorical', cat_pipe, cat_selector)

# ── ColumnTransformer ─────────────────────────────────────────────
preprocessor = ColumnTransformer(
    [cat_tuple, num_tuple],
    verbose_feature_names_out=False,
)

print("Preprocessing pipeline created.")


## 6. Logistic Regression Model Pipeline

**Logistic Regression** is the classification model taught in this course.  
It uses the **sigmoid function** to map any input value into a probability between 0 and 1, then classifies based on whether that probability is above or below 0.5.

We combine the preprocessor and the model into a single `pipeline` using `make_pipeline` — the same pattern shown in class.  
This ensures that the same preprocessing steps are applied consistently to both training and test data.


In [ ]:
model    = LogisticRegression()
pipeline = make_pipeline(preprocessor, model)

pipeline.fit(X_train, y_train)
print("Pipeline trained successfully.")


## 7. Make Predictions

In [ ]:
train_preds = pipeline.predict(X_train)
test_preds  = pipeline.predict(X_test)

print(f"Predicted (first 15 test): {test_preds[:15]}")
print(f"Actual    (first 15 test): {np.array(y_test[:15])}")


## 8. Confusion Matrix

The confusion matrix categorizes every prediction into one of four groups:

| | Predicted On Time (0) | Predicted Delayed (1) |
|---|---|---|
| **Actually On Time (0)** | True Negative (TN) ✅ | False Positive (FP) ❌ |
| **Actually Delayed (1)** | False Negative (FN) ❌ | True Positive (TP) ✅ |

We use `ConfusionMatrixDisplay.from_predictions()` exactly as shown in the course slides.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_preds,
    display_labels=['On Time (0)', 'Delayed (1)'],
    colorbar=True,
    ax=ax,
)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Evaluation Metrics

We compute four classification metrics as taught in class:

| Metric | Formula | What it measures |
|---|---|---|
| **Accuracy** | Correct / Total | Overall fraction of correct predictions |
| **Recall** | TP / (TP + FN) | Of all actual delayed days, how many did we catch? |
| **Precision** | TP / (TP + FP) | When we predict delayed, how often are we right? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of Precision and Recall |


In [ ]:
accuracy  = accuracy_score(y_test, test_preds)
recall    = recall_score(y_test, test_preds)
precision = precision_score(y_test, test_preds)
f1        = f1_score(y_test, test_preds)

print("=" * 42)
print("       MODEL EVALUATION — TEST DATA")
print("=" * 42)
print(f"  Accuracy   :  {accuracy:.4f}   ({accuracy * 100:.2f}%)")
print(f"  Recall     :  {recall:.4f}")
print(f"  Precision  :  {precision:.4f}")
print(f"  F1-Score   :  {f1:.4f}")
print("=" * 42)


## 10. Classification Report

`classification_report()` gives a full summary of precision, recall, F1-score, and support for each class — the same summary shown in the course slides.


In [ ]:
print("Classification Report: Test Data")
print(classification_report(y_test, test_preds, target_names=['On Time (0)', 'Delayed (1)']))


## 11. Detect Underfit or Overfit

As taught in class, we check for underfit and overfit by evaluating the model on **both** the training and test sets and comparing the results:

- **Underfit:** low quality on both train and test
- **Overfit:** significant gap between train (high) and test (low) quality
- **Good model:** acceptable quality on both, with no large gap


In [ ]:
train_accuracy = accuracy_score(y_train, train_preds)
test_accuracy  = accuracy_score(y_test,  test_preds)

print("Classification Report: Train Data")
print(classification_report(y_train, train_preds, target_names=['On Time (0)', 'Delayed (1)']))

print("Classification Report: Test Data")
print(classification_report(y_test, test_preds, target_names=['On Time (0)', 'Delayed (1)']))


In [ ]:
print(f"Train Accuracy : {train_accuracy:.4f}")
print(f"Test  Accuracy : {test_accuracy:.4f}")
print(f"Gap            : {abs(train_accuracy - test_accuracy):.4f}")

if abs(train_accuracy - test_accuracy) > 0.10:
    print("\n=> Overfit detected: significant gap between train and test accuracy.")
elif train_accuracy < 0.70 and test_accuracy < 0.70:
    print("\n=> Underfit detected: low quality on both train and test.")
else:
    print("\n=> Good model: acceptable quality on both train and test, no significant gap.")


## 12. Delay Rate by Country

We visualize the proportion of delayed airport-days per country using a bar plot.  
This shows which countries have structurally higher delay rates and provides real-world context for the model's predictions.


In [ ]:
country_names = {
    'FR': 'France', 'DE': 'Germany', 'IT': 'Italy',
    'ES': 'Spain',  'NL': 'Netherlands', 'GB': 'United Kingdom',
}

country_delay = (
    df.groupby('iso_country')['Is_Delayed']
    .agg(Delay_Rate='mean', Records='count')
    .reset_index()
    .sort_values('Delay_Rate', ascending=False)
)
country_delay['Country']      = country_delay['iso_country'].map(country_names)
country_delay['Delay Rate %'] = (country_delay['Delay_Rate'] * 100).round(1)

plt.figure(figsize=(9, 5))
sns.barplot(data=country_delay, x='Country', y='Delay Rate %',
            hue='Country', palette='Blues_r', legend=False)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Delay Rate (%)', fontsize=12)
plt.title('Proportion of Delayed Airport-Days by Country', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/delay_rate_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

print(country_delay[['Country', 'Delay Rate %', 'Records']].to_string(index=False))


## 13. Delay Rate by Season

We also explore how delay rates differ across seasons using a count plot — the same `sns.countplot()` technique taught in the Data Visualization chapter.


In [ ]:
plt.figure(figsize=(8, 5))
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
sns.countplot(data=df, x='Season', hue='Is_Delayed', order=season_order,
              palette={0: '#5499C7', 1: '#E74C3C'})
plt.xlabel('Season', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Delayed vs. On-Time Records by Season', fontsize=13)
plt.legend(title='Is Delayed', labels=['On Time (0)', 'Delayed (1)'])
plt.tight_layout()
plt.savefig('../figures/delay_by_season.png', dpi=150, bbox_inches='tight')
plt.show()


## 14. Discussion

### 14.1 Model Performance

The Logistic Regression model was trained on over 32,000 airport-day records and evaluated on more than 8,000 unseen test records.

The classification report shows the full breakdown of precision, recall, and F1-score for each class.  
By comparing train and test metrics, we can determine whether the model generalises well or shows signs of overfit or underfit.

### 14.2 Why Logistic Regression

As covered in the course slides, Logistic Regression is the standard binary classification algorithm — it uses the sigmoid function to produce probabilities and is well-suited to problems with a 0/1 target like `Is_Delayed`.

### 14.3 Country and Season Patterns

The bar plot shows that delay rates differ noticeably across the 6 countries.  
The count plot by season suggests that some seasons (e.g. Summer or Winter) tend to have more delayed records, likely due to higher traffic volume or adverse weather.

### 14.4 Limitations

| Limitation | Explanation |
|---|---|
| **Aggregated data** | Rows represent airport-days, not individual flights. The model predicts whether an airport will be delayed on average — not whether a specific flight will be late. |
| **No weather features** | Weather is one of the leading causes of delays but is not included in this dataset. |
| **EUROCONTROL coverage only** | Results apply to 6 European countries and may not generalise elsewhere. |
| **Linear decision boundary** | Logistic Regression assumes a linear relationship between features and the log-odds of the target. Non-linear patterns might be missed. |
| **15-minute threshold** | The cutoff for `Is_Delayed` follows industry convention but is somewhat arbitrary. |


## 15. Conclusion

This project built a complete data science pipeline for flight delay prediction across six European countries.

**Summary of Phase 4:**

1. Loaded the cleaned dataset from Phase 3 (40,000+ airport-day records, 21 columns)
2. Defined the binary target `Is_Delayed` (1 = delay > 15 min average, 0 = on time)
3. Built a preprocessing pipeline using `ColumnTransformer` with separate numeric and categorical sub-pipelines — exactly as taught in class
4. Trained a Logistic Regression classifier wrapped in a `make_pipeline`
5. Evaluated with accuracy, recall, precision, F1-score, confusion matrix, and classification report
6. Compared train vs. test quality to check for underfit or overfit
7. Visualized delay rates by country and by season to provide real-world context

**Key takeaway:** Airport-level delay patterns are predictable from historical punctuality data.  
Logistic Regression provides a clear, interpretable baseline model for this binary classification task.


## 16. Final Results Summary

In [ ]:
summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Recall', 'Precision', 'F1-Score'],
    'Value':  [f'{accuracy:.4f}', f'{recall:.4f}', f'{precision:.4f}', f'{f1:.4f}'],
})

print("=" * 38)
print("      FINAL MODEL PERFORMANCE")
print("=" * 38)
print(summary_df.to_string(index=False))
print("=" * 38)
print(f"Algorithm  : Logistic Regression")
print(f"Train rows : {len(X_train):,}  |  Test rows: {len(X_test):,}")
print(f"Features   : {X.shape[1]}")
print(f"Countries  : FR, DE, IT, ES, NL, GB")
print(f"Target     : Is_Delayed  (threshold = 15 min)")
